# CIOdD-UCR PDF Intelligence Pipeline — V5.5
## Research & RAG Edition · Google Colab

**Cambios principales respecto a V5.4:**
- **Detección de headings más estricta**: máximo 16 palabras, rechaza bloques con apariencia de oración (punto interno + >10 palabras).
- **Menos fusión de títulos**: solo une títulos adyacentes cortos (≤6 y ≤8 palabras); evita concatenar título + subtítulo o párrafos.
- **Limpieza de títulos**: elimina caracteres especiales (▒, ■, etc.) y normaliza espacios.
- **Filtro de tablas residuales**: rechaza tablas con `page_area_share < 0.03` (elimina fragmentos basura de página 20, etc.).
- **Descarte de chunks muy cortos**: elimina chunks con menos de 20 palabras únicas (títulos de portada aislados).
- Mantiene todas las mejoras de V5.4 (cobertura exacta 1.0, rechazo de páginas completas como tablas).

**Instrucciones:**
1. Ejecuta la celda de instalación.
2. Si hubo problemas con Pillow, reinicia el runtime.
3. Sube tus PDFs a `/content/PDFs`.
4. Ajusta la configuración y ejecuta.


## 1. Instalación de dependencias

In [ ]:
!pip install -q pymupdf pdfplumber pypdf pandas numpy pyarrow duckdb pytesseract scikit-learn joblib tqdm langdetect
!pip install -q "pillow>=10.0,<12"
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-spa tesseract-ocr-eng
print("✅ Dependencias instaladas")
print("⚠️ Si hubo problemas con Pillow, reinicia el runtime antes de continuar.")


## 2. Código completo del pipeline V5.5

In [ ]:
# -*- coding: utf-8 -*-
import os, sys, re, io, json, math, time, hashlib, statistics, unicodedata, logging
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

import pymupdf, pdfplumber, pypdf, pandas as pd, numpy as np, duckdb, pytesseract
from PIL import Image
from tqdm.auto import tqdm

try:
    from langdetect import detect
except Exception:
    detect = lambda x: "unknown"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("pdf_intelligence_v55")

@dataclass
class PipelineConfig:
    pipeline_version: str = "5.5"
    schema_version: str = "5.5.0"
    pdf_directory: Path = field(default_factory=lambda: Path("/content/PDFs"))
    output_directory: Path = field(default_factory=lambda: Path("/content/pdf_intelligence"))
    use_drive: bool = False
    drive_base: Path = field(default_factory=lambda: Path("/content/drive/MyDrive/pdf_intelligence"))
    max_documents: Optional[int] = None
    max_pages_per_doc: Optional[int] = None
    checkpoint_every: int = 1
    ocr_enabled: bool = True
    ocr_language: str = "spa+eng"
    ocr_dpi: int = 200
    ocr_min_chars: int = 80
    ocr_min_words: int = 20
    ocr_min_native_quality: float = 0.35
    heading_font_ratio: float = 1.18
    heading_min_score: float = 2.0
    heading_repeat_min_pages: int = 4
    heading_repeat_min_share: float = 0.08
    header_footer_threshold: float = 0.35
    header_footer_edge_lines: int = 3
    chunk_target_words: int = 550
    chunk_max_words: int = 800
    chunk_min_words: int = 60
    chunk_overlap_words: int = 80
    chunk_coverage_min: float = 0.995
    # Table filters (V5.4 stricter)
    table_lines_min_filled: float = 0.25
    table_lines_max_fragment: float = 0.50
    table_text_min_filled: float = 0.50
    table_text_max_fragment: float = 0.20
    table_text_max_lower_start: float = 0.25
    table_text_max_rows: int = 25
    table_text_max_cols: int = 8
    table_text_max_area: float = 0.70
    # V5.4: reject full-page "tables" that are actually prose
    table_lines_max_area: float = 0.85
    table_lines_max_avg_words: float = 15.0
    # V5.5: reject tiny residual tables and improve heading quality
    table_min_area: float = 0.03
    heading_max_words: int = 16
    min_chunk_words_discard: int = 20
    build_tfidf: bool = True
    generate_embeddings: bool = False
    embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    embedding_batch_size: int = 16

    def setup_paths(self):
        if self.use_drive:
            try:
                from google.colab import drive
                drive.mount("/content/drive")
                self.output_directory = self.drive_base
                self.pdf_directory = self.drive_base / "PDFs"
            except Exception as e:
                logger.warning(f"No se pudo montar Drive: {e}")
        self.output_directory.mkdir(parents=True, exist_ok=True)
        self.pdf_directory.mkdir(parents=True, exist_ok=True)
        return self

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def normalize_unicode(text):
    if not text: return ""
    text = unicodedata.normalize("NFKC", text)
    return text.replace("\u00ad", "").replace("\xa0", " ")

def normalize_text(text):
    if not text: return ""
    text = normalize_unicode(text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines).strip()

def single_line(text):
    return re.sub(r"\s+", " ", normalize_unicode(text or "")).strip()

def clean_title(text):
    """V5.5: remove box-drawing / special chars and normalize spacing in titles."""
    if not text: return ""
    text = re.sub(r"[▒■□▪▫●○◆◇★☆►▶◀◄→←↑↓]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    # collapse repeated punctuation
    text = re.sub(r"([.:;])\1+", r"\1", text)
    return text

def word_count(text):
    return len(re.findall(r"\b[\wÀ-ÿ]+\b", text or "", flags=re.UNICODE))

def sha256_file(path):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            sha.update(chunk)
    return sha.hexdigest()

def stable_id(prefix, value):
    digest = hashlib.sha256(value.encode("utf-8", errors="ignore")).hexdigest()[:16]
    return f"{prefix}_{digest}"

def detect_language(text):
    text = (text or "").strip()
    if len(text) < 80: return "unknown"
    try: return detect(text[:5000])
    except Exception: return "unknown"

def write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

def normalize_pattern(line):
    return re.sub(r"\d+", "#", single_line(line).lower())

def sha256_text(text):
    return hashlib.sha256((text or "").encode("utf-8", errors="ignore")).hexdigest()

def flatten_row(row):
    result = {}
    for key, value in row.items():
        if isinstance(value, (dict, list, tuple)):
            result[key] = json.dumps(value, ensure_ascii=False, default=str)
        else:
            result[key] = value
    return result

class QualityAssessor:
    @staticmethod
    def alphanumeric_ratio(text):
        if not text: return 0.0
        chars = [c for c in text if not c.isspace()]
        if not chars: return 0.0
        valid = sum(c.isalnum() or c in ".,;:!?¿¡()[]{}%$€₡+-/='\"" for c in chars)
        return valid / len(chars)

    @staticmethod
    def replacement_ratio(text):
        return (text or "").count("\ufffd") / max(len(text or ""), 1)

    def text_quality(self, text):
        wc = word_count(text)
        density = min(1.0, math.log1p(wc) / math.log1p(350)) if wc else 0.0
        encoding = max(0.0, 1.0 - min(1.0, self.replacement_ratio(text) * 50))
        ratio = self.alphanumeric_ratio(text)
        legibility = max(0.0, min(1.0, (ratio - 0.40) / 0.45))
        tokens = re.findall(r"\S+", text or "")
        token_structure = max(0.0, 1.0 - (sum(len(t) > 40 for t in tokens) / len(tokens) * 4)) if tokens else 0.0
        score = 0.35 * density + 0.25 * encoding + 0.25 * legibility + 0.15 * token_structure
        return {
            "density_score": round(density, 4),
            "encoding_score": round(encoding, 4),
            "legibility_score": round(legibility, 4),
            "token_structure_score": round(token_structure, 4),
            "text_quality_score": round(score, 4),
        }

QUALITY = QualityAssessor()

class PDFProfiler:
    @staticmethod
    def classify(document):
        n_pages = len(document)
        if n_pages == 0: return "empty"
        sample = sorted({int(round(i * (n_pages - 1) / 6)) for i in range(7)}) if n_pages > 7 else list(range(n_pages))
        with_text = sum(1 for idx in sample if word_count(normalize_text(document[idx].get_text("text", sort=True))) >= 20)
        if with_text == len(sample): return "digital"
        if with_text == 0: return "scanned"
        return "hybrid"

    def profile(self, pdf_path):
        result = {
            "filename": Path(pdf_path).name,
            "file_size_bytes": os.path.getsize(pdf_path),
            "sha256": sha256_file(pdf_path),
            "title": "", "author": "", "subject": "", "creator": "", "producer": "",
            "pdf_creation_date": "", "page_count": 0, "encrypted": False, "profile_type": "unknown",
        }
        try:
            reader = pypdf.PdfReader(pdf_path)
            result["page_count"] = len(reader.pages)
            result["encrypted"] = bool(reader.is_encrypted)
            metadata = reader.metadata or {}
            for target, source in {"title": "/Title", "author": "/Author", "subject": "/Subject",
                                   "creator": "/Creator", "producer": "/Producer",
                                   "pdf_creation_date": "/CreationDate"}.items():
                result[target] = str(metadata.get(source, "") or "")
        except Exception as e:
            result["metadata_error"] = str(e)
        try:
            document = pymupdf.open(pdf_path)
            result["profile_type"] = self.classify(document)
            document.close()
        except Exception as e:
            result["profile_error"] = str(e)
        return result

class LayoutExtractor:
    def extract(self, page):
        data = page.get_text("dict", sort=True)
        blocks, images, font_sizes = [], [], []
        for block_index, block in enumerate(data.get("blocks", [])):
            block_type = block.get("type", 0)
            if block_type == 1:
                images.append({"block_id": block_index, "bbox": list(block.get("bbox", [])), "width": block.get("width"), "height": block.get("height")})
                continue
            if block_type != 0: continue
            spans, lines_out = [], []
            for line in block.get("lines", []):
                line_parts = []
                for span in line.get("spans", []):
                    text = single_line(span.get("text", ""))
                    if not text: continue
                    font_size = float(span.get("size", 0) or 0)
                    font_sizes.append(font_size)
                    spans.append({"text": text, "font": str(span.get("font", "")), "font_size": font_size, "flags": int(span.get("flags", 0) or 0), "bbox": list(span.get("bbox", []))})
                    line_parts.append(text)
                line_text = single_line(" ".join(line_parts))
                if line_text: lines_out.append(line_text)
            block_text = "\n".join(lines_out).strip()
            if block_text:
                blocks.append({"block_id": block_index, "bbox": list(block.get("bbox", [])), "text": block_text, "spans": spans})
        median_font = statistics.median(font_sizes) if font_sizes else 0
        return {"blocks": blocks, "median_font_size": median_font, "images": images}

LAYOUT = LayoutExtractor()

class HeadingDetector:
    standard = re.compile(r"^(resumen( ejecutivo)?|abstract|executive summary|summary|key (messages|findings)|introducci[oó]n|introduction|antecedentes|background|marco te[oó]rico|metodolog[ií]a|methodology|m[eé]todos?|methods?|resultados|results?|discusi[oó]n|discussion|conclusiones|conclusions?|recomendaciones|recommendations|referencias|references|bibliograf[ií]a|bibliography|anexos?|annex(es)?|appendix|appendices|glosario|glossary|prefacio|preface|foreword|acknowledg(e)?ments?|agradecimientos|[ií]ndice|contents|table of contents)$", re.I)
    prefixed = re.compile(r"^(annex|appendix|chapter|part|section|anexo|ap[eé]ndice|cap[ií]tulo|parte|secci[oó]n)\s+[A-Z0-9IVXivx]{1,4}\b", re.I)
    numbered = re.compile(r"^\s*(?:(\d{1,2}(?:\.\d{1,2}){0,3})[\.)]?|([IVXLC]{1,6})[\.\)])\s+(?=[A-ZÁÉÍÓÚÑ¿¡])")
    footnote = re.compile(r"^\s*\d{1,3}\s*(see|ibid|source|sources|note|available|cf\.|op\.|http|www|v[eé]ase|ver\b|fuente)", re.I)
    bullet = re.compile(r"^\s*[•▪■◦●·*\-–—]\s+")
    caption = re.compile(r"^\s*(figure|fig\.|table|tabla|figura|cuadro|chart|gr[aá]fico|source|fuente|note|nota)\s*\d", re.I)
    bare_int = re.compile(r"^\s*\d{1,3}\s+\S")

    @staticmethod
    def is_bold(span):
        font = (span.get("font", "") or "").lower()
        return any(k in font for k in ["bold", "black", "semibold", "demi", "heavy"])

    @staticmethod
    def _main_font_size(spans):
        if not spans: return 0.0
        best = max(spans, key=lambda s: len(s.get("text", "") or ""))
        return float(best.get("font_size", 0) or 0)

    def detect(self, block, median_font, page_width=None, page_height=None, page_number=1):
        result = {"is_heading": False, "heading_confidence": 0.0, "heading_level": None, "heading_signals": []}
        text = single_line(block.get("text", ""))
        words = text.split()
        n = len(words)
        if not text or n > 16 or len(text) > 180: return result  # V5.5 stricter
        alnum = [c for c in text if c.isalnum()]
        if len(alnum) < 3 or sum(c.isdigit() for c in alnum) / len(alnum) > 0.35: return result
        if self.bullet.match(text) or self.footnote.match(text) or self.caption.match(text): return result
        if text.count(")") > text.count("(") or (self.bare_int.match(text) and "," in text): return result
        if text[-1] in ",;" or (text[-1] == "." and n > 6): return result
        is_standard = bool(self.standard.match(text.strip(" .:-"))) or bool(self.prefixed.match(text))
        is_numbered = bool(self.numbered.match(text)) and n <= 16
        # V5.5: reject long sentence-like blocks that are not numbered/standard
        if n > 10 and "." in text[:-1] and not is_numbered and not is_standard: return result
        if words[0][:1].islower() and not is_numbered: return result
        if len(text) < 5 and not is_standard: return result
        spans = block.get("spans", []) or []
        main_size = self._main_font_size(spans)
        ratio = (main_size / median_font) if median_font else 1.0
        if median_font and ratio < 0.92 and not is_standard: return result
        bbox = block.get("bbox") or []
        if page_height and len(bbox) == 4 and page_number > 1 and n <= 14 and not is_standard:
            center = (bbox[1] + bbox[3]) / 2.0
            if center < 0.05 * page_height or center > 0.93 * page_height: return result
        bold_ratio = (sum(self.is_bold(s) for s in spans) / len(spans)) if spans else 0.0
        is_upper = text.isupper() and 2 <= n <= 14 and len(text) >= 6
        title_case = (2 <= n <= 12 and sum(w[:1].isupper() for w in words) >= max(2, int(n * 0.6)))
        lines = len((block.get("text", "") or "").splitlines())
        score, signals = 0.0, []
        if is_standard: score += 3.0; signals.append("standard")
        if is_numbered: score += 2.0; signals.append("numbered")
        if ratio >= 1.30: score += 2.5; signals.append("font>=1.30")
        elif ratio >= 1.18: score += 1.5; signals.append("font>=1.18")
        if bold_ratio >= 0.70: score += 1.0; signals.append("bold")
        if is_upper: score += 1.0; signals.append("upper")
        if title_case: score += 0.5; signals.append("title_case")
        if lines and lines <= 2: score += 0.3; signals.append("short_block")
        if text[-1] not in ".:": score += 0.3; signals.append("no_terminal_period")
        if score < 2.0: return result
        depth = None
        if is_numbered:
            m = self.numbered.match(text)
            if m and m.group(1): depth = m.group(1).count(".") + 1
        level = depth or (1 if ratio >= 1.6 else 2 if ratio >= 1.30 else 3)
        result.update({"is_heading": True, "heading_confidence": round(min(1.0, score / 4.0), 3), "heading_level": int(level), "heading_signals": signals})
        return result

HEADING = HeadingDetector()

def demote_repeated_headings(pages, config):
    n_pages = len(pages)
    if n_pages < 8: return 0
    seen = defaultdict(set)
    for page in pages:
        for block in page.get("blocks", []):
            if block.get("is_heading"):
                seen[single_line(block.get("text", "")).lower()].add(page["page_number"])
    threshold = max(config.heading_repeat_min_pages, math.ceil(n_pages * config.heading_repeat_min_share))
    repeated = {k for k, v in seen.items() if len(v) >= threshold}
    demoted = 0
    for page in pages:
        for block in page.get("blocks", []):
            if block.get("is_heading") and single_line(block.get("text", "")).lower() in repeated:
                block["is_heading"] = False
                block["heading_confidence"] = 0.0
                block["heading_demoted"] = "repeated_on_many_pages"
                demoted += 1
        page["heading_count"] = sum(bool(b.get("is_heading")) for b in page.get("blocks", []))
    return demoted

class OCRExtractor:
    def __init__(self, config):
        self.config = config
    def needs_ocr(self, text, quality):
        return (len(text.strip()) < self.config.ocr_min_chars or word_count(text) < self.config.ocr_min_words or quality < self.config.ocr_min_native_quality)
    def extract(self, page):
        zoom = self.config.ocr_dpi / 72
        pixmap = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False)
        image = Image.open(io.BytesIO(pixmap.tobytes("png")))
        try:
            dataframe = pytesseract.image_to_data(image, lang=self.config.ocr_language, output_type=pytesseract.Output.DATAFRAME)
        except Exception:
            dataframe = pytesseract.image_to_data(image, lang="eng", output_type=pytesseract.Output.DATAFRAME)
        dataframe = dataframe[dataframe["text"].notna()].copy()
        dataframe["text"] = dataframe["text"].astype(str).map(single_line)
        dataframe = dataframe[dataframe["text"] != ""]
        if dataframe.empty: return {"text": "", "confidence": 0}
        confidence = pd.to_numeric(dataframe["conf"], errors="coerce")
        valid_confidence = confidence[confidence >= 0]
        grouping = ["block_num", "par_num", "line_num"]
        lines = dataframe.groupby(grouping, sort=False)["text"].apply(lambda s: " ".join(s)).tolist()
        text = normalize_text("\n".join(lines))
        mean_confidence = float(valid_confidence.mean()) / 100 if len(valid_confidence) else 0
        return {"text": text, "confidence": round(mean_confidence, 4)}

class AdaptiveTableExtractor:
    def __init__(self, config):
        self.config = config
        self.strategies = [
            ("lines", {"vertical_strategy": "lines", "horizontal_strategy": "lines", "snap_tolerance": 3, "join_tolerance": 3, "intersection_tolerance": 3}),
            ("text", {"vertical_strategy": "text", "horizontal_strategy": "text", "min_words_vertical": 3, "min_words_horizontal": 1, "text_x_tolerance": 3, "text_y_tolerance": 3}),
        ]
        self.caption_re = re.compile(r"^(table|tabla|cuadro|figure|figura|chart|gr[aá]fico|annex|anexo)\s*[\dA-Z]", re.I)
    @staticmethod
    def clean_matrix(table):
        out = []
        for row in (table or []):
            clean = [single_line("" if c is None else str(c)) for c in row]
            if any(clean): out.append(clean)
        return out
    @staticmethod
    def features(matrix, bbox, page_area):
        cells = [c for r in matrix for c in r]
        filled = [c for c in cells if c.strip()]
        nf = max(1, len(filled))
        area = 0.0
        if bbox and len(bbox) == 4 and page_area:
            area = max(0.0, (bbox[2] - bbox[0])) * max(0.0, (bbox[3] - bbox[1])) / page_area
        return {
            "filled_ratio": round(len(filled) / max(1, len(cells)), 3),
            "fragment_ratio": round(sum(len(c.strip()) <= 2 for c in filled) / nf, 3),
            "lower_start_ratio": round(sum(c.strip()[:1].islower() for c in filled) / nf, 3),
            "avg_words_per_cell": round(sum(len(c.split()) for c in filled) / nf, 2),
            "page_area_share": round(area, 3),
        }
    def judge(self, strategy, matrix, feats):
        rows, cols = len(matrix), max(len(r) for r in matrix)
        if rows < 2 or cols < 2:
            return False, "menos de 2 filas o 2 columnas"
        # V5.5: reject tiny residual fragments
        if feats.get("page_area_share", 1.0) < self.config.table_min_area:
            return False, "área demasiado pequeña (residuo)"
        # V5.4: reject full-page prose misdetected as table (lines strategy)
        if strategy == "lines":
            if feats["page_area_share"] > self.config.table_lines_max_area and feats["avg_words_per_cell"] > self.config.table_lines_max_avg_words:
                return False, "página completa con prosa (falso positivo)"
            if feats["filled_ratio"] < self.config.table_lines_min_filled:
                return False, "celdas mayormente vacías"
            if feats["fragment_ratio"] >= self.config.table_lines_max_fragment:
                return False, "celdas fragmentadas"
            return True, "ok"
        # text strategy
        if rows > self.config.table_text_max_rows or cols > self.config.table_text_max_cols:
            return False, "rejilla de texto demasiado grande"
        if feats["lower_start_ratio"] >= self.config.table_text_max_lower_start:
            return False, "celdas inician en minúscula"
        if feats["fragment_ratio"] >= self.config.table_text_max_fragment:
            return False, "celdas fragmentadas"
        if feats["filled_ratio"] < self.config.table_text_min_filled:
            return False, "celdas mayormente vacías"
        if feats["page_area_share"] >= self.config.table_text_max_area:
            return False, "ocupa casi toda la página"
        return True, "ok"
    @staticmethod
    def to_markdown(matrix, max_rows=40):
        def esc(c):
            return (c or "").replace("|", "\\|").replace("\n", " ")
        width = max(len(r) for r in matrix)
        rows = [r + [""] * (width - len(r)) for r in matrix[:max_rows]]
        lines = ["| " + " | ".join(esc(c) for c in rows[0]) + " |", "|" + "|".join(["---"] * width) + "|"]
        lines += ["| " + " | ".join(esc(c) for c in r) + " |" for r in rows[1:]]
        if len(matrix) > max_rows:
            lines.append(f"| … ({len(matrix) - max_rows} filas más) |" + " |" * (width - 1))
        return "\n".join(lines)
    def find_caption(self, page, bbox):
        try:
            y0 = max(0.0, bbox[1] - 45.0)
            if bbox[1] - y0 < 8: return ""
            text = page.crop((0, y0, page.width, bbox[1])).extract_text() or ""
            for line in reversed([single_line(x) for x in text.splitlines() if single_line(x)]):
                if self.caption_re.match(line): return line[:300]
        except Exception:
            pass
        return ""
    def extract(self, page, document_id, page_number):
        accepted, rejected, seen = [], [], set()
        try:
            page_area = float(page.width) * float(page.height)
        except Exception:
            page_area = 0.0
        accepted_bboxes = []
        for strategy_name, settings in self.strategies:
            try:
                found = page.find_tables(table_settings=settings)
            except Exception:
                found = []
            for table_object in found:
                try:
                    matrix = self.clean_matrix(table_object.extract())
                except Exception:
                    continue
                if not matrix: continue
                n_cols = max(len(r) for r in matrix)
                matrix = [r + [""] * (n_cols - len(r)) for r in matrix]
                try:
                    bbox = [round(float(v), 2) for v in table_object.bbox]
                except Exception:
                    bbox = []
                fingerprint = hashlib.sha256(json.dumps(matrix, ensure_ascii=False).encode("utf-8")).hexdigest()
                if fingerprint in seen: continue
                seen.add(fingerprint)
                feats = self.features(matrix, bbox, page_area)
                ok, reason = self.judge(strategy_name, matrix, feats)
                if bbox and any(self._iou(bbox, ab) > 0.6 for ab in accepted_bboxes):
                    continue
                headers_raw = matrix[0]
                counter, headers = {}, []
                for i, h in enumerate(headers_raw):
                    base = h if h else f"column_{i + 1}"
                    counter[base] = counter.get(base, 0) + 1
                    headers.append(base if counter[base] == 1 else f"{base}_{counter[base]}")
                record = {
                    "table_id": stable_id("tbl", f"{document_id}|{page_number}|{strategy_name}|{bbox}|{matrix[:3]}"),
                    "document_id": document_id, "page_number": page_number, "strategy": strategy_name,
                    "bbox": bbox, "rows": len(matrix) - 1, "columns": n_cols,
                    "table_confidence": ("high" if strategy_name == "lines" and feats["filled_ratio"] >= 0.5 else "medium" if ok else "low"),
                    "quality_features": feats, "caption": self.find_caption(page, bbox) if bbox else "",
                    "headers": headers, "records": [dict(zip(headers, r)) for r in matrix[1:]], "matrix": matrix,
                }
                if ok:
                    record["markdown"] = self.to_markdown(matrix)
                    accepted.append(record)
                    if bbox: accepted_bboxes.append(bbox)
                else:
                    record["reject_reason"] = reason
                    record.pop("records", None)
                    rejected.append(record)
        return {"accepted": accepted, "rejected": rejected}
    @staticmethod
    def _iou(box1, box2):
        if not box1 or not box2 or len(box1) != 4 or len(box2) != 4: return 0.0
        x_left = max(box1[0], box2[0]); y_top = max(box1[1], box2[1])
        x_right = min(box1[2], box2[2]); y_bottom = min(box1[3], box2[3])
        if x_right < x_left or y_bottom < y_top: return 0.0
        inter = (x_right - x_left) * (y_bottom - y_top)
        a1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
        a2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union = a1 + a2 - inter
        return inter / union if union > 0 else 0.0

def infer_repeated_lines(pages, config):
    if len(pages) < 3: return set()
    counter = Counter()
    k = config.header_footer_edge_lines
    for page in pages:
        text = page.get("native_text") or page.get("ocr_text") or page.get("clean_text") or ""
        lines = [single_line(x) for x in text.splitlines() if single_line(x)]
        edge = lines[:k] + lines[-k:]
        for pattern in {normalize_pattern(x) for x in edge if 1 <= len(x) <= 140}:
            counter[pattern] += 1
    minimum = max(3, math.ceil(len(pages) * config.header_footer_threshold))
    return {p for p, c in counter.items() if c >= minimum and p.strip()}

def is_repeated_line(line, patterns):
    norm = normalize_pattern(line)
    if norm in patterns: return True
    tokens = single_line(line).split()
    return bool(tokens) and all(normalize_pattern(t) in patterns for t in tokens)

def remove_repeated_lines(text, patterns, config):
    if not patterns or not text: return normalize_text(text), []
    lines = text.splitlines()
    k = config.header_footer_edge_lines
    keep, removed = [], []
    last = len(lines) - 1
    for i, line in enumerate(lines):
        at_edge = i < k or i > last - k
        if at_edge and single_line(line) and is_repeated_line(line, patterns):
            removed.append(single_line(line))
        else:
            keep.append(line)
    return normalize_text("\n".join(keep)), removed

def clean_blocks(blocks, patterns, page_height):
    if not patterns: return blocks
    cleaned = []
    for block in blocks:
        bbox = block.get("bbox") or []
        in_edge = bool(page_height and len(bbox) == 4 and (bbox[3] < 0.12 * page_height or bbox[1] > 0.88 * page_height))
        if in_edge and is_repeated_line(" ".join(block.get("text", "").split()), patterns):
            continue
        if in_edge:
            kept = [ln for ln in block.get("text", "").splitlines() if not is_repeated_line(ln, patterns)]
            if not kept: continue
            block = dict(block)
            block["text"] = "\n".join(kept)
        cleaned.append(block)
    return cleaned

class SectionBuilder:
    def build(self, document_id, pages):
        sections, state = [], {"current": None, "counter": 0, "last_bottom": None, "last_page": None}
        def start(title, page_number, source, conf=0.0, level=None):
            state["counter"] += 1
            return {"section_id": f"{document_id}_S{state['counter']:04d}", "document_id": document_id, "title": title, "title_source": source, "heading_confidence": conf, "heading_level": level, "page_start": page_number, "page_end": page_number, "paragraphs": []}
        def close():
            cur = state["current"]
            if cur and cur["paragraphs"]:
                sections.append(cur)
            state["current"] = None
        for page in pages:
            pn = page["page_number"]
            blocks = page.get("blocks", []) or []
            if not blocks or page.get("extraction_source") == "ocr":
                text = page.get("clean_text", "")
                if not text: continue
                if state["current"] is None:
                    state["current"] = start("Contenido inicial", pn, "none")
                state["current"]["paragraphs"].append({"page_number": pn, "text": text, "block_id": None})
                state["current"]["page_end"] = pn
                state["last_bottom"] = None
                continue
            for block in blocks:
                text = normalize_text(block.get("text", ""))
                if not text: continue
                bbox = block.get("bbox") or [0, 0, 0, 0]
                if block.get("is_heading"):
                    title = clean_title(single_line(text))
                    conf = float(block.get("heading_confidence", 0) or 0)
                    level = block.get("heading_level")
                    cur = state["current"]
                    if cur is not None and not cur["paragraphs"]:
                        adjacent = (state["last_page"] == pn and state["last_bottom"] is not None and (bbox[1] - state["last_bottom"]) < 18)
                        # V5.5: only merge short adjacent titles (avoid title+subtitle fusion)
                        if adjacent and len(title.split()) <= 6 and len(cur["title"].split()) <= 8:
                            cur["title"] = (cur["title"] + " " + title)[:300]
                            cur["heading_confidence"] = max(cur["heading_confidence"], conf)
                        else:
                            close()
                            state["current"] = start(title[:300], pn, "detected", conf, level)
                    else:
                        close()
                        state["current"] = start(title[:300], pn, "detected", conf, level)
                    state["last_bottom"], state["last_page"] = bbox[3], pn
                else:
                    if state["current"] is None:
                        state["current"] = start("Contenido inicial", pn, "none")
                    state["current"]["paragraphs"].append({"page_number": pn, "text": text, "block_id": block.get("block_id")})
                    state["current"]["page_end"] = pn
                    state["last_bottom"] = None
        close()
        for section in sections:
            section["section_title"] = section["title"]
            section["text"] = normalize_text("\n\n".join(p["text"] for p in section["paragraphs"]))
            section["word_count"] = word_count(section["text"])
            section["paragraph_count"] = len(section["paragraphs"])
            section["language"] = detect_language(section["text"])
        return sections

def split_long_paragraph(text, max_words):
    if word_count(text) <= max_words: return [text]
    sentences = re.split(r"(?<=[\.\?\!;:])\s+(?=[A-ZÁÉÍÓÚÑ¿¡0-9\"“(])", text)
    parts, cur, n = [], [], 0
    for sentence in sentences:
        w = word_count(sentence)
        if w > max_words:
            if cur: parts.append(" ".join(cur)); cur, n = [], 0
            tokens = sentence.split()
            for i in range(0, len(tokens), max_words):
                parts.append(" ".join(tokens[i:i + max_words]))
            continue
        if cur and n + w > max_words:
            parts.append(" ".join(cur)); cur, n = [], 0
        cur.append(sentence); n += w
    if cur: parts.append(" ".join(cur))
    return parts

class ChunkBuilder:
    def __init__(self, config):
        self.config = config
    def build(self, document_id, filename, sections, tables_by_page, quality_by_page, label_by_page):
        chunks, state = [], {"n": 0}
        def pages_info(page_set):
            tables = sorted({t for p in page_set for t in tables_by_page.get(p, [])})
            qs = [quality_by_page[p] for p in page_set if p in quality_by_page]
            return tables, round(float(np.mean(qs)) if qs else 0.0, 4)
        def refresh(chunk):
            chunk["text"] = normalize_text(chunk["text"])
            chunk["word_count"] = word_count(chunk["text"])
            chunk["text_sha256"] = sha256_text(chunk["text"].lower())
            pages = chunk["_pages"]
            chunk["page_start"], chunk["page_end"] = min(pages), max(pages)
            chunk["page_label_start"] = label_by_page.get(chunk["page_start"], "")
            chunk["page_label_end"] = label_by_page.get(chunk["page_end"], "")
            chunk["related_table_ids"], chunk["quality_score"] = pages_info(pages)
        def emit(section, items, overlap_words):
            # V5.4: only count non-overlap items for unique_words
            new_items = [i for i in items if not i.get("overlap")]
            if not new_items: return None
            state["n"] += 1
            chunk = {
                "chunk_id": f"{document_id}_C{state['n']:05d}", "document_id": document_id, "filename": filename,
                "section_id": section["section_id"], "section_title": section["title"],
                "section_title_source": section.get("title_source", "none"),
                "section_title_reliable": bool(section.get("title_source") == "detected" and float(section.get("heading_confidence", 0) or 0) >= 0.6),
                "merged_section_ids": [], "merged_section_titles": [], "short_section": False,
                "text": "\n\n".join(i["text"] for i in items), "overlap_words": overlap_words,
                "unique_words": sum(word_count(i["text"]) for i in new_items),
                "language": detect_language("\n".join(i["text"] for i in items)),
                "_pages": {i["page"] for i in new_items},
            }
            refresh(chunk)
            chunks.append(chunk)
            return chunk
        def absorb(chunk, new_items, extra_section=None):
            chunk["text"] = chunk["text"] + "\n\n" + "\n\n".join(i["text"] for i in new_items)
            chunk["unique_words"] += sum(word_count(i["text"]) for i in new_items)
            chunk["_pages"] |= {i["page"] for i in new_items}
            if extra_section:
                chunk["merged_section_ids"].append(extra_section["section_id"])
                if extra_section.get("title_source") == "detected":
                    chunk["merged_section_titles"].append(extra_section["title"])
            refresh(chunk)
        for section in sections:
            paragraphs = []
            for p in section.get("paragraphs", []):
                text = normalize_text(p.get("text", ""))
                if text:
                    for part in split_long_paragraph(text, self.config.chunk_max_words):
                        paragraphs.append({"text": part, "page": p["page_number"]})
            if not paragraphs: continue
            section_words = sum(word_count(p["text"]) for p in paragraphs)
            if section_words < self.config.chunk_min_words:
                # V5.4: do NOT inject title into unique_words for short sections
                items = paragraphs  # title is NOT added to items for unique_words
                add_words = sum(word_count(i["text"]) for i in items)
                if chunks and chunks[-1]["word_count"] + add_words <= self.config.chunk_max_words:
                    absorb(chunks[-1], items, section)
                else:
                    c = emit(section, items, 0)
                    if c is not None: c["short_section"] = True
                continue
            items, new_count, cur_words = [], 0, 0
            def flush_and_overlap(next_page):
                nonlocal items, new_count, cur_words
                c = emit(section, items, sum(word_count(i["text"]) for i in items if i.get("overlap")))
                items, new_count, cur_words = [], 0, 0
                if c is not None and self.config.chunk_overlap_words:
                    tail = c["text"].split()[-self.config.chunk_overlap_words:]
                    if tail:
                        items = [{"text": " ".join(tail), "page": next_page, "overlap": True}]
                        cur_words = len(tail)
            for para in paragraphs:
                w = word_count(para["text"])
                if new_count and cur_words + w > self.config.chunk_max_words:
                    flush_and_overlap(para["page"])
                items.append(para); new_count += 1; cur_words += w
                if cur_words >= self.config.chunk_target_words:
                    flush_and_overlap(para["page"])
            if new_count:
                tail_words = sum(word_count(i["text"]) for i in items if not i.get("overlap"))
                last = chunks[-1] if chunks else None
                new_only = [i for i in items if not i.get("overlap")]
                if (tail_words < self.config.chunk_min_words and last is not None and last["section_id"] == section["section_id"] and last["unique_words"] + tail_words <= self.config.chunk_max_words * 1.25):
                    absorb(last, new_only)
                else:
                    emit(section, items, sum(word_count(i["text"]) for i in items if i.get("overlap")))
        for chunk in chunks:
            chunk.pop("_pages", None)
        # V5.5: drop extremely short chunks (title-only noise)
        min_w = getattr(self.config, "min_chunk_words_discard", 20)
        chunks = [c for c in chunks if c.get("unique_words", 0) >= min_w or c.get("word_count", 0) >= min_w]
        return chunks

class DocumentExtractor:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.profiler = PDFProfiler()
        self.ocr = OCRExtractor(config)
        self.tables = AdaptiveTableExtractor(config)
    def extract(self, pdf_path):
        profile = self.profiler.profile(pdf_path)
        document_id = stable_id("doc", profile["sha256"])
        document = {"document_id": document_id, **profile, "processed_at": utc_now(), "pipeline_version": self.config.pipeline_version}
        pages, tables, rejected, errors = [], [], [], []
        pdf = pymupdf.open(pdf_path)
        plumber_pdf = None
        try:
            try:
                plumber_pdf = pdfplumber.open(pdf_path)
            except Exception as e:
                errors.append({"document_id": document_id, "stage": "pdfplumber_open", "error": str(e)})
            page_limit = min(len(pdf), self.config.max_pages_per_doc or len(pdf))
            for page_index in range(page_limit):
                page_number = page_index + 1
                page = pdf[page_index]
                try:
                    native_text = normalize_text(page.get_text("text", sort=True))
                    native_metrics = QUALITY.text_quality(native_text)
                    layout = LAYOUT.extract(page)
                    blocks, median_font = layout["blocks"], layout["median_font_size"]
                    heading_count = 0
                    for block in blocks:
                        info = HEADING.detect(block, median_font, float(page.rect.width), float(page.rect.height), page_number)
                        block.update(info)
                        heading_count += int(info["is_heading"])
                    final_text, extraction_source = native_text, "native"
                    ocr_applied, ocr_text, ocr_confidence = False, "", None
                    if self.config.ocr_enabled and self.ocr.needs_ocr(native_text, native_metrics["text_quality_score"]):
                        ocr_result = self.ocr.extract(page)
                        ocr_applied = True
                        ocr_text, ocr_confidence = ocr_result["text"], ocr_result["confidence"]
                        if (word_count(ocr_text) > max(1, word_count(native_text)) * 1.20 or word_count(native_text) < self.config.ocr_min_words):
                            final_text, extraction_source = ocr_text, "ocr"
                    page_tables = []
                    if plumber_pdf is not None:
                        try:
                            found = self.tables.extract(plumber_pdf.pages[page_index], document_id, page_number)
                            page_tables = found["accepted"]
                            rejected.extend(found["rejected"])
                        except Exception as e:
                            errors.append({"document_id": document_id, "page_number": page_number, "stage": "tables", "error": str(e)})
                    tables.extend(page_tables)
                    final_metrics = QUALITY.text_quality(final_text)
                    words = word_count(final_text)
                    flags = []
                    if extraction_source == "ocr":
                        flags.append("ocr")
                        if (ocr_confidence or 0) < 0.60: flags.append("ocr_low_confidence")
                    if words == 0: flags.append("empty")
                    elif words < 20: flags.append("low_text")
                    if not blocks: flags.append("no_layout_blocks")
                    if layout["images"] and words < 30: flags.append("image_page")
                    try: label = page.get_label() or ""
                    except Exception: label = ""
                    pages.append({
                        "page_id": f"{document_id}_P{page_number:05d}", "document_id": document_id, "filename": profile["filename"],
                        "page_number": page_number, "page_label": label, "width": float(page.rect.width), "height": float(page.rect.height),
                        "native_text": native_text, "ocr_text": ocr_text, "clean_text": normalize_text(final_text),
                        "extraction_source": extraction_source, "ocr_applied": ocr_applied, "ocr_confidence": ocr_confidence,
                        "word_count": words, "language": detect_language(final_text), "block_count": len(blocks),
                        "heading_count": heading_count, "image_count": len(layout["images"]), "table_count": len(page_tables),
                        "page_flags": flags, "blocks_source": "native" if extraction_source == "native" else "ocr_text_only",
                        "blocks": blocks, "images_metadata": layout["images"], **final_metrics,
                    })
                except Exception as e:
                    errors.append({"document_id": document_id, "page_number": page_number, "stage": "page", "error": str(e)})
                    pages.append({"page_id": f"{document_id}_P{page_number:05d}", "document_id": document_id, "filename": profile["filename"], "page_number": page_number, "page_label": "", "clean_text": "", "word_count": 0, "blocks": [], "page_flags": ["page_error"], "error": str(e)})
        finally:
            pdf.close()
            if plumber_pdf: plumber_pdf.close()
        repeated = infer_repeated_lines(pages, self.config)
        for page in pages:
            height = page.get("height")
            page["clean_text"], removed = remove_repeated_lines(page.get("clean_text", ""), repeated, self.config)
            page["removed_edge_lines"] = sorted(set(removed))
            page["word_count"] = word_count(page["clean_text"])
            page["blocks"] = clean_blocks(page.get("blocks", []), repeated, height)
        demoted = demote_repeated_headings(pages, self.config)
        return {"document": document, "pages": pages, "tables": tables, "tables_rejected": rejected, "errors": errors, "repeated_lines": sorted(repeated), "headings_demoted": demoted}

class CorpusBuilder:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.documents, self.pages, self.sections = [], [], []
        self.chunks, self.tables, self.tables_rejected = [], [], []
        self.errors, self.log = [], []
        self.checkpoint_path = config.output_directory / "checkpoint.json"
        self.extractor = DocumentExtractor(config)
        self.section_builder = SectionBuilder()
        self.chunk_builder = ChunkBuilder(config)
    def load_checkpoint(self):
        if self.checkpoint_path.exists():
            try:
                data = json.loads(self.checkpoint_path.read_text(encoding="utf-8"))
                logger.info(f"Checkpoint: {len(data.get('processed', []))} documentos ya procesados")
                return set(data.get("processed", []))
            except Exception:
                pass
        return set()
    def save_checkpoint(self, processed):
        write_json(self.checkpoint_path, {"processed": list(processed), "saved_at": utc_now()})
    def process(self):
        processed = self.load_checkpoint()
        pdfs = sorted(
            p for p in self.config.pdf_directory.rglob("*")
            if p.is_file() and p.suffix.lower() == ".pdf" and p.name not in processed
        )
        if self.config.max_documents: pdfs = pdfs[:self.config.max_documents]
        logger.info(f"Procesando {len(pdfs)} PDFs nuevos")
        for i, pdf_path in enumerate(tqdm(pdfs, desc="Documentos")):
            start = time.time()
            try:
                result = self.extractor.extract(str(pdf_path))
                document, pages = result["document"], result["pages"]
                document_id = document["document_id"]
                sections = self.section_builder.build(document_id, pages)
                tables_by_page = defaultdict(list)
                for t in result["tables"]:
                    tables_by_page[t["page_number"]].append(t["table_id"])
                quality_by_page = {p["page_number"]: float(p.get("text_quality_score", 0) or 0) for p in pages}
                label_by_page = {p["page_number"]: p.get("page_label", "") for p in pages}
                chunks = self.chunk_builder.build(document_id, document["filename"], sections, tables_by_page, quality_by_page, label_by_page)
                section_words = sum(word_count(p["text"]) for s in sections for p in s["paragraphs"])
                chunk_unique = sum(c["unique_words"] for c in chunks)
                n_pages = max(1, len(pages))
                ocr_pages = sum(p.get("extraction_source") == "ocr" for p in pages)
                low_text = sum("low_text" in p.get("page_flags", []) or "empty" in p.get("page_flags", []) for p in pages)
                document.update({
                    "extracted_pages": len(pages), "total_words": sum(p.get("word_count", 0) for p in pages),
                    "total_tables": len(result["tables"]), "tables_rejected": len(result["tables_rejected"]),
                    "total_sections": len(sections), "sections_with_detected_title": sum(s.get("title_source") == "detected" for s in sections),
                    "total_chunks": len(chunks), "ocr_pages": ocr_pages, "low_text_pages": low_text,
                    "headings_demoted": result["headings_demoted"], "section_words": section_words,
                    "chunk_unique_words": chunk_unique, "chunk_coverage": round(chunk_unique / section_words, 4) if section_words else 1.0,
                    "average_extraction_quality": round(float(np.mean([float(p.get("text_quality_score", 0) or 0) for p in pages])), 4) if pages else 0,
                    "profile_type_observed": ("scanned" if ocr_pages / n_pages >= 0.8 else "digital" if ocr_pages == 0 and low_text / n_pages < 0.05 else "hybrid"),
                    "repeated_header_footer_patterns": result["repeated_lines"],
                })
                self.documents.append(document); self.pages.extend(pages); self.sections.extend(sections)
                self.chunks.extend(chunks); self.tables.extend(result["tables"]); self.tables_rejected.extend(result["tables_rejected"])
                self.errors.extend(result["errors"])
                processed.add(pdf_path.name)
                if (i + 1) % self.config.checkpoint_every == 0:
                    self.save_checkpoint(processed)
                self.log.append({"document_id": document_id, "filename": document["filename"], "status": "ok", "seconds": round(time.time() - start, 3), "pages": len(pages), "chunks": len(chunks), "chunk_coverage": document["chunk_coverage"]})
            except Exception as e:
                logger.error(f"Error en {pdf_path.name}: {e}")
                self.errors.append({"filename": pdf_path.name, "stage": "document", "error": str(e), "timestamp": utc_now()})
                self.log.append({"filename": pdf_path.name, "status": "failed", "error": str(e)})
        logger.info("Exportando corpus...")
        self.export_base()
        self.build_duckdb()
        if self.config.build_tfidf and self.chunks: self.build_tfidf()
        if self.config.generate_embeddings and self.chunks: self.build_embeddings()
        self.check_integrity()
        self.write_reports()
        logger.info(f"✅ Pipeline V{self.config.pipeline_version} completado")
    def export_base(self):
        for name, rows in [("documents", self.documents), ("pages", self.pages), ("sections", self.sections), ("chunks", self.chunks), ("tables", self.tables)]:
            write_jsonl(self.config.output_directory / f"{name}.jsonl", rows)
            if rows:
                pd.DataFrame([flatten_row(r) for r in rows]).to_parquet(self.config.output_directory / f"{name}.parquet", index=False)
        write_jsonl(self.config.output_directory / "tables_rejected.jsonl", self.tables_rejected)
        write_jsonl(self.config.output_directory / "processing_log.jsonl", self.log)
        write_jsonl(self.config.output_directory / "errors.jsonl", self.errors)
    def build_duckdb(self):
        path = self.config.output_directory / "corpus.duckdb"
        if path.exists(): path.unlink()
        con = duckdb.connect(str(path))
        try:
            for name in ["documents", "pages", "sections", "chunks", "tables"]:
                parquet = self.config.output_directory / f"{name}.parquet"
                if parquet.exists():
                    safe = str(parquet).replace("'", "''")
                    con.execute(f"CREATE TABLE {name} AS SELECT * FROM read_parquet('{safe}')")
        finally:
            con.close()
    def build_tfidf(self):
        import joblib
        from sklearn.feature_extraction.text import TfidfVectorizer
        texts = [(c.get("section_title", "") if c.get("section_title_reliable") else "") + "\n" + c.get("text", "") for c in self.chunks]
        vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), max_df=0.95, min_df=1, max_features=100_000, sublinear_tf=True, norm="l2")
        matrix = vectorizer.fit_transform(texts)
        joblib.dump(vectorizer, self.config.output_directory / "tfidf_vectorizer.joblib")
        joblib.dump(matrix, self.config.output_directory / "tfidf_matrix.joblib")
    def build_embeddings(self):
        try:
            from sentence_transformers import SentenceTransformer
            import faiss
            model = SentenceTransformer(self.config.embedding_model)
            texts = [(c.get("section_title", "") if c.get("section_title_reliable") else "") + "\n" + c.get("text", "") for c in self.chunks]
            embeddings = np.asarray(model.encode(texts, batch_size=self.config.embedding_batch_size, show_progress_bar=True, normalize_embeddings=True), dtype="float32")
            np.save(self.config.output_directory / "embeddings.npy", embeddings)
            index = faiss.IndexFlatIP(embeddings.shape[1])
            index.add(embeddings)
            faiss.write_index(index, str(self.config.output_directory / "faiss.index"))
        except Exception as e:
            logger.warning(f"Embeddings no generados: {e}")
    def check_integrity(self):
        checks = []
        def add(name, ok, detail="", severity="FAIL"):
            checks.append({"check": name, "status": "PASS" if ok else severity, "detail": detail})
        def unique(rows, key):
            vals = [r.get(key) for r in rows]
            return len(vals) == len(set(vals))
        add("ids únicos", all([unique(self.documents, "document_id"), unique(self.pages, "page_id"), unique(self.sections, "section_id"), unique(self.chunks, "chunk_id"), unique(self.tables, "table_id")]))
        low_cov = [(d["filename"], d["chunk_coverage"]) for d in self.documents if d.get("chunk_coverage", 1) < self.config.chunk_coverage_min]
        add(f"cobertura >= {self.config.chunk_coverage_min}", not low_cov, str(low_cov))
        status = "FAIL" if any(c["status"] == "FAIL" for c in checks) else "PASS_WITH_WARNINGS" if any(c["status"] == "WARNING" for c in checks) else "PASS"
        write_json(self.config.output_directory / "integrity_report.json", {"generated_at": utc_now(), "status": status, "checks": checks})
    def write_reports(self):
        write_json(self.config.output_directory / "quality_report.json", {"generated_at": utc_now(), "pipeline_version": self.config.pipeline_version, "documents": len(self.documents), "pages": len(self.pages), "chunks": len(self.chunks), "tables": len(self.tables)})
        write_json(self.config.output_directory / "manifest.json", {"pipeline": "CIOdD-UCR PDF Intelligence Pipeline", "version": self.config.pipeline_version, "generated_at": utc_now(), "configuration": asdict(self.config)})

print("✅ Pipeline V5.5 cargado")

## 3. Configuración y ejecución

In [ ]:
import os
from pathlib import Path
from google.colab import files

config = PipelineConfig(
    max_documents=None,
    max_pages_per_doc=None,
    generate_embeddings=False,
    use_drive=False,
)
config.setup_paths()

print("PDF directory:", config.pdf_directory)
print("Output directory:", config.output_directory)

# --- Subir PDFs si la carpeta está vacía ---
pdfs_found = list(config.pdf_directory.rglob("*"))
pdfs_found = [p for p in pdfs_found if p.is_file() and p.suffix.lower() == ".pdf"]
print(f"PDFs encontrados: {len(pdfs_found)}")
for p in pdfs_found:
    print(" -", p.name)

if not pdfs_found:
    print("\nNo hay PDFs en /content/PDFs.")
    print("Selecciona los archivos a subir...")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = config.pdf_directory / name
        dest.write_bytes(data)
        print("Guardado:", dest)
    pdfs_found = [p for p in config.pdf_directory.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"]
    print(f"PDFs listos: {len(pdfs_found)}")

# --- Opcional: borrar checkpoint para reprocesar todo ---
checkpoint = config.output_directory / "checkpoint.json"
if checkpoint.exists():
    print("\nCheckpoint existente detectado.")
    # Descomenta la siguiente línea si quieres forzar reprocesamiento
    # checkpoint.unlink()
    # print("Checkpoint eliminado. Se reprocesarán todos los PDFs.")

builder = CorpusBuilder(config)
builder.process()

print("\n=== RESUMEN ===")
print(f"Documentos procesados: {len(builder.documents)}")
print(f"Chunks totales: {len(builder.chunks)}")
print(f"Tablas aceptadas: {len(builder.tables)}")
print(f"Tablas rechazadas: {len(builder.tables_rejected)}")


## 4. Descargar resultados (opcional)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/pdf_intelligence_V55", "zip", "/content/pdf_intelligence")
files.download("/content/pdf_intelligence_V55.zip")
print("✅ Archivo zip descargado")
